# DemandPulse

## Inventory Optimization & Decision Intelligence

**Notebook 05 — From Forecasts to Inventory Decisions**

This notebook converts the outputs of the prior DemandPulse research notebooks into explicit inventory decisions.

It uses:

- point forecasts and baseline results from Notebook 01,
- probabilistic P10/P50/P90 forecasts from Notebook 02,
- causal promotion estimates from Notebook 03,
- hierarchical point forecasts from Notebook 04.

The objective is **decision quality**, not another forecasting benchmark.

> **Important:** FreshRetailNet-50K provides normalized demand rather than retailer accounting data. Every financial quantity in this notebook is therefore a **clearly labelled simulation assumption expressed in normalized cost units**, not a claimed real business value.

No frontend, API, production deployment, or replenishment service is built here.

## 1. Decision model and scientific boundary

The notebook uses a transparent single-period perishable-inventory approximation for each SKU/day forecast, plus a lead-time reorder-point summary across the first configurable forecast days.

For an order quantity `q` and realized demand `d`:

- fulfilled demand = `min(q, d)`
- shortage = `max(d - q, 0)`
- leftover = `max(q - d, 0)`
- waste units = `leftover × waste_rate`

The simulated economics include procurement, holding, shortage, waste, fixed ordering cost and sales revenue.

This is intentionally not presented as the retailer's true economics. The notebook keeps every cost assumption configurable and runs sensitivity/scenario analysis because the dataset does not supply the required financial inputs.

In [ ]:
%pip install -q "datasets>=3.0" "pandas>=2.0" "numpy>=1.24" "pyarrow>=14" "matplotlib>=3.7" "tqdm>=4.66"

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import random
import warnings
from dataclasses import dataclass, asdict, replace
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import datasets
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("Python:", platform.python_version())
print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## 2. Configuration

`SERIES_SAMPLE_PERCENT` intentionally matches the prior notebooks. Set it to `100` for full-data experiments after confirming memory/runtime capacity.

In [ ]:
DATASET_ID = "Dingdong-Inc/FreshRetailNet-50K"
SERIES_KEYS = ["store_id", "product_id"]
SERIES_SAMPLE_PERCENT = 30
STREAM_BATCH_SIZE = 50_000
FORECAST_HORIZON = 7
LEAD_TIME_DAYS = 3

# Decision simulation controls.
N_MONTE_CARLO = 1_000
SCENARIO_SAMPLE_ROWS = 500
WASTE_RATE = 0.20

# Stress-test multipliers are simulation controls, not measured retailer facts.
STOCKOUT_RISK_COST_MULTIPLIER = 2.0
HIGH_HOLDING_COST_MULTIPLIER = 3.0

REPORTS_DIR = Path("reports")
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = Path("models")
for directory in [REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")

## 3. Configurable cost model — simulation assumptions only

The defaults below are **not business values**. They are normalized illustrative assumptions used to demonstrate how forecast uncertainty becomes a decision.

The selling price is normalized to `1.0` cost unit per demand unit. Other costs are expressed relative to that scale.

Change these values before using the framework for a real retailer.

In [ ]:
@dataclass(frozen=True)
class InventoryCostConfig:
    unit_cost: float = 0.55
    selling_price: float = 1.00
    holding_cost: float = 0.03
    stockout_cost: float = 0.40
    waste_cost: float = 0.15
    ordering_cost: float = 0.05
    service_level_target: float = 0.90
    waste_rate: float = WASTE_RATE

    def validate(self) -> None:
        nonnegative = [
            "unit_cost", "selling_price", "holding_cost", "stockout_cost",
            "waste_cost", "ordering_cost", "waste_rate",
        ]
        for name in nonnegative:
            value = getattr(self, name)
            if not np.isfinite(value) or value < 0:
                raise ValueError(f"{name} must be finite and nonnegative; got {value}")
        if not 0 < self.service_level_target < 1:
            raise ValueError("service_level_target must be strictly between 0 and 1.")
        if not 0 <= self.waste_rate <= 1:
            raise ValueError("waste_rate must lie in [0, 1].")

COST = InventoryCostConfig()
COST.validate()

cost_assumptions = pd.DataFrame({
    "parameter": list(asdict(COST).keys()),
    "simulation_value": list(asdict(COST).values()),
    "status": "SIMULATION ASSUMPTION — NOT RETAILER ACCOUNTING DATA",
})
display(cost_assumptions)

## 4. Previous-notebook artifact audit

Notebook 05 does not depend on prior kernel memory. It reads saved artifacts from disk.

Scientifically required inputs are:

- Notebook 02's row-level probabilistic predictions,
- Notebook 03's causal effect estimates,
- Notebook 04's hierarchical SKU forecasts.

Notebook 01's comparison table is loaded for context when available.

In [ ]:
ARTIFACTS = {
    "nb01_final_comparison": TABLES_DIR / "final_model_comparison.csv",
    "nb02_predictions": TABLES_DIR / "probabilistic_eval_predictions.parquet",
    "nb02_final_metrics": TABLES_DIR / "probabilistic_final_metrics.csv",
    "nb03_ate": TABLES_DIR / "ate_estimates.csv",
    "nb03_hte": TABLES_DIR / "heterogeneous_treatment_effects.csv",
    "nb03_cate_segments": TABLES_DIR / "cate_segment_analysis.csv",
    "nb03_sensitivity": TABLES_DIR / "causal_sensitivity_analysis.csv",
    "nb04_hierarchical_forecasts": TABLES_DIR / "hierarchical_forecasts.csv",
    "nb04_reconciliation": TABLES_DIR / "reconciliation_results.csv",
    "nb04_level_metrics": TABLES_DIR / "hierarchy_level_metrics.csv",
}

artifact_audit = pd.DataFrame([
    {
        "artifact": name,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    }
    for name, path in ARTIFACTS.items()
])
display(artifact_audit)

required = ["nb02_predictions", "nb03_ate", "nb03_hte", "nb04_hierarchical_forecasts"]
missing_required = [name for name in required if not ARTIFACTS[name].exists()]
if missing_required:
    raise FileNotFoundError(
        "Notebook 05 requires saved outputs from Notebooks 02–04. Missing: "
        + ", ".join(missing_required)
        + ". Run the preceding notebooks top-to-bottom first; no prior kernel memory is required."
    )

In [ ]:
prob_preds = pd.read_parquet(ARTIFACTS["nb02_predictions"])
ate_estimates = pd.read_csv(ARTIFACTS["nb03_ate"])
hte = pd.read_csv(ARTIFACTS["nb03_hte"])
hier_forecasts = pd.read_csv(ARTIFACTS["nb04_hierarchical_forecasts"])

nb01_final = pd.read_csv(ARTIFACTS["nb01_final_comparison"]) if ARTIFACTS["nb01_final_comparison"].exists() else pd.DataFrame()
nb02_final = pd.read_csv(ARTIFACTS["nb02_final_metrics"]) if ARTIFACTS["nb02_final_metrics"].exists() else pd.DataFrame()
nb03_cate = pd.read_csv(ARTIFACTS["nb03_cate_segments"]) if ARTIFACTS["nb03_cate_segments"].exists() else pd.DataFrame()
nb03_sensitivity = pd.read_csv(ARTIFACTS["nb03_sensitivity"]) if ARTIFACTS["nb03_sensitivity"].exists() else pd.DataFrame()
nb04_reconciliation = pd.read_csv(ARTIFACTS["nb04_reconciliation"]) if ARTIFACTS["nb04_reconciliation"].exists() else pd.DataFrame()
nb04_levels = pd.read_csv(ARTIFACTS["nb04_level_metrics"]) if ARTIFACTS["nb04_level_metrics"].exists() else pd.DataFrame()

for frame, date_cols in [
    (prob_preds, ["origin_date", "target_date"]),
    (hier_forecasts, ["target_date"]),
]:
    for col in date_cols:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors="raise")

print("Probabilistic predictions:", prob_preds.shape)
print("Hierarchical forecasts:", hier_forecasts.shape)
if not nb01_final.empty:
    print("Notebook 01 point-forecast benchmark context:")
    display(nb01_final)
if not nb04_levels.empty:
    print("Notebook 04 hierarchy-level benchmark context:")
    display(nb04_levels)
display(ate_estimates)

## 5. Forecast-source rules

The inventory policies deliberately use different information sources:

1. **Historical Average** — mean demand from the sampled training history.
2. **Point Forecast** — Notebook 04's pre-specified `MinT-Style` SKU point forecast. It is not selected using Notebook 05 outcomes.
3. **P50** — Notebook 02's median forecast.
4. **P90** — Notebook 02's conservative upper quantile.
5. **Optimized Decision** — a cost/service-level constrained quantile decision from the P10/P50/P90 distribution approximation.

Using a fixed hierarchical method avoids selecting a reconciliation method by looking at the same official-eval outcomes used later for inventory evaluation.

In [ ]:
EXPECTED_PROB_COLS = {
    "store_id", "product_id", "target_date", "horizon", "target",
    "p_p10", "p_p50", "p_p90",
}
missing = EXPECTED_PROB_COLS - set(prob_preds.columns)
if missing:
    raise ValueError(f"Notebook 02 prediction artifact is missing required columns: {sorted(missing)}")

sku_hier = hier_forecasts.loc[
    hier_forecasts["level"].astype(str).eq("SKU")
    & hier_forecasts["method"].astype(str).eq("MinT-Style")
].copy()
if sku_hier.empty:
    raise ValueError("Notebook 04 artifact has no SKU / MinT-Style forecasts.")

# SKU node IDs are created by Notebook 04 as SKU|city|store|third_category|product.
parts = sku_hier["node_id"].astype(str).str.split("|", expand=True)
if parts.shape[1] != 5:
    raise ValueError("Unexpected Notebook 04 SKU node_id format; expected five pipe-separated fields.")
sku_hier["city_id_hier"] = parts[1]
sku_hier["store_id_hier"] = parts[2]
sku_hier["third_category_id_hier"] = parts[3]
sku_hier["product_id_hier"] = parts[4]
sku_hier = sku_hier[[
    "store_id_hier", "product_id_hier", "third_category_id_hier", "city_id_hier",
    "target_date", "horizon", "forecast",
]].rename(columns={"forecast": "hier_point_forecast"})

# Stable string keys avoid dtype mismatches across CSV/Parquet artifacts.
def stable_id_key(s: pd.Series) -> pd.Series:
    return s.astype(str).str.replace(r"\.0$", "", regex=True)

prob_preds["store_key"] = stable_id_key(prob_preds["store_id"])
prob_preds["product_key"] = stable_id_key(prob_preds["product_id"])
sku_hier["store_key"] = stable_id_key(sku_hier["store_id_hier"])
sku_hier["product_key"] = stable_id_key(sku_hier["product_id_hier"])

prob_preds = prob_preds.merge(
    sku_hier[["store_key", "product_key", "target_date", "horizon", "hier_point_forecast", "city_id_hier", "third_category_id_hier"]],
    on=["store_key", "product_key", "target_date", "horizon"],
    how="left",
    validate="one_to_one",
)

missing_hier = prob_preds["hier_point_forecast"].isna().sum()
print(f"Rows without matching MinT SKU point forecast: {missing_hier:,}")

## 6. Lightweight remote context from FreshRetailNet-50K

Notebook 05 uses the same remote dataset and the exact deterministic series-selection rule from the earlier notebooks.

Only compact historical summaries and official-eval context are materialized. No same-day stock variable is used to choose an order quantity; realized stock information is retained only for after-the-fact risk diagnostics.

In [ ]:
def series_selected(store_id, product_id, percent: float = SERIES_SAMPLE_PERCENT) -> bool:
    """Exact deterministic complete-series sampling used throughout DemandPulse."""
    if not 0 < percent <= 100:
        raise ValueError("percent must lie in (0, 100].")
    key = f"{store_id}-{product_id}".encode("utf-8")
    digest = hashlib.sha256(key).digest()
    bucket = int.from_bytes(digest[:8], "big") % 10_000
    return bucket < int(percent * 100)

# Determinism sanity test.
assert series_selected("1", "2", 30) == series_selected("1", "2", 30)
assert series_selected("1", "2", 100)

train_stream = load_dataset(DATASET_ID, split="train", streaming=True)
eval_stream = load_dataset(DATASET_ID, split="eval", streaming=True)
print(type(train_stream), type(eval_stream))

In [ ]:
HISTORY_COLUMNS = ["store_id", "product_id", "dt", "sale_amount", "stock_hour6_22_cnt"]
EVAL_CONTEXT_COLUMNS = [
    "city_id", "store_id", "management_group_id", "first_category_id", "second_category_id", "third_category_id",
    "product_id", "dt", "sale_amount", "activity_flag", "discount", "holiday_flag", "stock_hour6_22_cnt",
]


def _selected_batch_mask(batch: Mapping[str, Sequence]) -> np.ndarray:
    return np.fromiter(
        (series_selected(s, p) for s, p in zip(batch["store_id"], batch["product_id"])),
        dtype=bool,
        count=len(batch["store_id"]),
    )


def compute_training_history_summary(stream, batch_size: int = STREAM_BATCH_SIZE) -> pd.DataFrame:
    """Stream training rows and return compact per-series statistics without materializing the full dataset."""
    partials: List[pd.DataFrame] = []
    processed = selected = 0
    for batch in tqdm(stream.iter(batch_size=batch_size), desc="Training history summary", unit="batch"):
        processed += len(batch["store_id"])
        mask = _selected_batch_mask(batch)
        selected += int(mask.sum())
        if not mask.any():
            continue
        frame = pd.DataFrame({c: np.asarray(batch[c], dtype=object)[mask] for c in HISTORY_COLUMNS})
        frame["sale_amount"] = pd.to_numeric(frame["sale_amount"], errors="raise")
        frame["stock_hour6_22_cnt"] = pd.to_numeric(frame["stock_hour6_22_cnt"], errors="coerce")
        frame["dt"] = pd.to_datetime(frame["dt"], errors="raise")
        frame["stockout"] = frame["stock_hour6_22_cnt"].fillna(0).gt(0).astype(int)
        g = frame.groupby(SERIES_KEYS, observed=True, as_index=False).agg(
            demand_sum=("sale_amount", "sum"),
            demand_count=("sale_amount", "size"),
            demand_sumsq=("sale_amount", lambda x: float(np.square(x.to_numpy(float)).sum())),
            zero_sales_count=("sale_amount", lambda x: int((x == 0).sum())),
            historical_stockout_count=("stockout", "sum"),
            history_min_date=("dt", "min"),
            history_max_date=("dt", "max"),
        )
        partials.append(g)
    if not partials:
        raise RuntimeError("No training rows were selected; check sampling configuration.")
    all_parts = pd.concat(partials, ignore_index=True)
    summary = all_parts.groupby(SERIES_KEYS, observed=True, as_index=False).agg(
        demand_sum=("demand_sum", "sum"),
        demand_count=("demand_count", "sum"),
        demand_sumsq=("demand_sumsq", "sum"),
        zero_sales_count=("zero_sales_count", "sum"),
        historical_stockout_count=("historical_stockout_count", "sum"),
        history_min_date=("history_min_date", "min"),
        history_max_date=("history_max_date", "max"),
    )
    summary["historical_mean"] = summary["demand_sum"] / summary["demand_count"].clip(lower=1)
    variance = (summary["demand_sumsq"] / summary["demand_count"].clip(lower=1)) - np.square(summary["historical_mean"])
    summary["historical_std"] = np.sqrt(np.maximum(variance, 0))
    summary["zero_sales_ratio"] = summary["zero_sales_count"] / summary["demand_count"].clip(lower=1)
    summary["historical_stockout_rate"] = summary["historical_stockout_count"] / summary["demand_count"].clip(lower=1)
    print(f"Training rows processed: {processed:,}; selected: {selected:,}; series: {len(summary):,}")
    return summary


def materialize_eval_context(stream, batch_size: int = STREAM_BATCH_SIZE) -> pd.DataFrame:
    chunks: List[pd.DataFrame] = []
    processed = selected = 0
    for batch in tqdm(stream.iter(batch_size=batch_size), desc="Eval context", unit="batch"):
        processed += len(batch["store_id"])
        mask = _selected_batch_mask(batch)
        selected += int(mask.sum())
        if not mask.any():
            continue
        chunks.append(pd.DataFrame({c: np.asarray(batch[c], dtype=object)[mask] for c in EVAL_CONTEXT_COLUMNS}))
    if not chunks:
        raise RuntimeError("No eval rows selected.")
    out = pd.concat(chunks, ignore_index=True)
    out["dt"] = pd.to_datetime(out["dt"], errors="raise")
    numeric = ["sale_amount", "activity_flag", "discount", "holiday_flag", "stock_hour6_22_cnt"]
    for c in numeric:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    print(f"Eval rows processed: {processed:,}; selected: {selected:,}; materialized: {len(out):,}")
    return out

history_summary = compute_training_history_summary(train_stream)
eval_context = materialize_eval_context(eval_stream)

## 7. Assemble the decision dataset

The official eval outcome is used only for **after-the-fact policy evaluation**. It is never used to calculate the order quantity.

The order quantity uses only prior-history summaries, forecast artifacts and configuration assumptions.

In [ ]:
history_summary["store_key"] = stable_id_key(history_summary["store_id"])
history_summary["product_key"] = stable_id_key(history_summary["product_id"])
eval_context["store_key"] = stable_id_key(eval_context["store_id"])
eval_context["product_key"] = stable_id_key(eval_context["product_id"])

context_cols = [
    "store_key", "product_key", "dt", "city_id", "management_group_id", "first_category_id",
    "second_category_id", "third_category_id", "activity_flag", "discount", "holiday_flag",
    "stock_hour6_22_cnt", "sale_amount",
]

decisions = prob_preds.merge(
    history_summary[[
        "store_key", "product_key", "historical_mean", "historical_std", "zero_sales_ratio",
        "historical_stockout_rate", "history_max_date",
    ]],
    on=["store_key", "product_key"], how="left", validate="many_to_one",
).merge(
    eval_context[context_cols].rename(columns={"dt": "target_date", "sale_amount": "eval_sale_amount"}),
    on=["store_key", "product_key", "target_date"], how="left", validate="one_to_one",
)

# The Notebook 02 target and remote eval sale should match where both are present.
match_mask = decisions["eval_sale_amount"].notna()
if match_mask.any():
    max_target_diff = float(np.abs(decisions.loc[match_mask, "target"].astype(float) - decisions.loc[match_mask, "eval_sale_amount"].astype(float)).max())
    print("Maximum Notebook02-vs-remote eval target difference:", max_target_diff)
    if max_target_diff > 1e-8:
        raise ValueError("Notebook 02 target values do not match the remote eval split.")

for c in ["p_p10", "p_p50", "p_p90", "target", "hier_point_forecast", "historical_mean"]:
    decisions[c] = pd.to_numeric(decisions[c], errors="coerce")

required_decision = ["target", "p_p10", "p_p50", "p_p90", "historical_mean", "hier_point_forecast"]
missing_rows = decisions[required_decision].isna().any(axis=1)
print(f"Decision rows before common mask: {len(decisions):,}")
print(f"Rows missing a required forecast/history input: {missing_rows.sum():,}")
decisions = decisions.loc[~missing_rows].copy()

if decisions.empty:
    raise RuntimeError("No common decision rows remain after merging prior artifacts and history.")

if (decisions[["p_p10", "p_p50", "p_p90"]].to_numpy() < 0).any():
    raise ValueError("Operational quantile predictions should be nonnegative in Notebook 02 artifact.")
if not ((decisions["p_p10"] <= decisions["p_p50"]) & (decisions["p_p50"] <= decisions["p_p90"])).all():
    raise ValueError("Notebook 02 operational quantiles are not monotonic on some rows.")

print("Common decision dataset:", decisions.shape)
print("Date range:", decisions["target_date"].min(), "to", decisions["target_date"].max())

## 8. Causal promotion integration without double counting

Notebook 02's production-like probabilistic model already used the **planned future activity flag** as a known feature. Therefore, adding an ATE/CATE again to those official eval forecasts would double count promotion information.

Notebook 05 follows two rules:

1. **Official eval policy comparison:** no causal uplift is added to P10/P50/P90.
2. **Counterfactual promotion scenario:** start from observations without promotion, then apply a causal effect distribution estimated in Notebook 03.

For an operational deterministic adjustment, the notebook uses a deliberately conservative rule:

- if the entire 95% CI is above zero, use the lower CI bound;
- if the entire 95% CI is below zero, use the upper CI bound;
- if the CI crosses zero, use `0` rather than pretending the effect is established.

Segment estimates are preferred in the order Store → Category → City → global AIPW when available.

In [ ]:
def _normalise_segment_value(value) -> str:
    if pd.isna(value):
        return "<NA>"
    return str(value).replace(".0", "") if str(value).endswith(".0") else str(value)

# Global AIPW fallback.
aipw_rows = ate_estimates.loc[ate_estimates["Method"].astype(str).str.upper().eq("AIPW")]
if aipw_rows.empty:
    raise ValueError("Notebook 03 ate_estimates.csv does not contain an AIPW row.")
global_effect = aipw_rows.iloc[0]
GLOBAL_CAUSAL = {
    "effect": float(global_effect["ATE"]),
    "ci_lower": float(global_effect["CI_Lower"]),
    "ci_upper": float(global_effect["CI_Upper"]),
    "source": "Global AIPW",
}

# Map HTE rows for only segments with explicit confidence intervals.
hte_maps: Dict[str, Dict[str, dict]] = {}
for seg_type in ["store_id", "first_category_id", "city_id"]:
    sub = hte.loc[hte["segment_type"].astype(str).eq(seg_type)].copy()
    mapping = {}
    for _, r in sub.iterrows():
        mapping[_normalise_segment_value(r["segment"])] = {
            "effect": float(r["estimated_effect"]),
            "ci_lower": float(r["CI_Lower"]),
            "ci_upper": float(r["CI_Upper"]),
            "source": f"HTE:{seg_type}",
        }
    hte_maps[seg_type] = mapping


def causal_effect_for_row(row: pd.Series) -> dict:
    for col in ["store_id", "first_category_id", "city_id"]:
        key = _normalise_segment_value(row.get(col, np.nan))
        if key in hte_maps.get(col, {}):
            return hte_maps[col][key]
    return GLOBAL_CAUSAL.copy()


def conservative_effect(effect: float, ci_lower: float, ci_upper: float) -> float:
    if ci_lower > 0:
        return float(ci_lower)
    if ci_upper < 0:
        return float(ci_upper)
    return 0.0

effect_records = [causal_effect_for_row(r) for _, r in decisions.iterrows()]
effects = pd.DataFrame(effect_records, index=decisions.index).rename(columns={
    "effect": "causal_effect_estimate",
    "ci_lower": "causal_ci_lower",
    "ci_upper": "causal_ci_upper",
    "source": "causal_effect_source",
})
decisions = pd.concat([decisions, effects], axis=1)
decisions["credible_causal_uplift"] = [
    conservative_effect(e, lo, hi)
    for e, lo, hi in zip(decisions["causal_effect_estimate"], decisions["causal_ci_lower"], decisions["causal_ci_upper"])
]

print("Global causal fallback:", GLOBAL_CAUSAL)
print("Rows with non-zero conservative causal uplift:", int(decisions["credible_causal_uplift"].ne(0).sum()))

## 9. Sparse probabilistic distribution from P10/P50/P90

P10/P50/P90 are **not a complete probability distribution**. For decision simulation only, the notebook constructs a transparent piecewise-linear quantile function through:

- an extrapolated nonnegative lower endpoint,
- P10,
- P50,
- P90,
- an extrapolated upper endpoint.

This approximation preserves the modeled quantiles but introduces a tail assumption. Results are therefore sensitivity-analysis outputs, not a claim that the true demand distribution has this exact shape.

In [ ]:
PROB_KNOTS = np.array([0.0, 0.10, 0.50, 0.90, 1.0], dtype=float)


def distribution_knots(p10: float, p50: float, p90: float) -> np.ndarray:
    if not (0 <= p10 <= p50 <= p90):
        raise ValueError(f"Invalid quantiles: {p10}, {p50}, {p90}")
    lower = max(0.0, p10 - (p50 - p10))
    upper = p90 + (p90 - p50)
    return np.array([lower, p10, p50, p90, upper], dtype=float)


def quantile_from_three(p10: float, p50: float, p90: float, probability: float) -> float:
    probability = float(np.clip(probability, 0, 1))
    return float(np.interp(probability, PROB_KNOTS, distribution_knots(p10, p50, p90)))


def distribution_mean_from_three(p10: float, p50: float, p90: float) -> float:
    y = distribution_knots(p10, p50, p90)
    # Integral of a piecewise-linear quantile function = trapezoidal area.
    return float(np.sum(np.diff(PROB_KNOTS) * (y[:-1] + y[1:]) / 2))


def cdf_from_three(value: float, p10: float, p50: float, p90: float) -> float:
    y = distribution_knots(p10, p50, p90)
    # Duplicate knot values can occur for zero/intermittent demand; preserve monotonicity.
    unique_y = np.unique(y)
    # For a flat quantile segment (common with zero/intermittent demand), the CDF at that
    # value is the largest probability whose quantile is still <= value.
    unique_p = np.array([PROB_KNOTS[y == uy].max() for uy in unique_y], dtype=float)
    if len(unique_y) == 1:
        return 1.0 if value >= unique_y[0] else 0.0
    return float(np.interp(value, unique_y, unique_p, left=0.0, right=1.0))


def sample_from_three(p10: float, p50: float, p90: float, n: int, rng: np.random.Generator) -> np.ndarray:
    u = rng.random(n)
    return np.maximum(np.interp(u, PROB_KNOTS, distribution_knots(p10, p50, p90)), 0.0)

# Quantile-function self-tests.
assert math.isclose(quantile_from_three(8, 10, 15, 0.10), 8)
assert math.isclose(quantile_from_three(8, 10, 15, 0.50), 10)
assert math.isclose(quantile_from_three(8, 10, 15, 0.90), 15)
assert distribution_mean_from_three(8, 10, 15) >= 0
assert 0 <= cdf_from_three(10, 8, 10, 15) <= 1
print("Probabilistic decision-function tests passed.")

## 10. Economic critical fractile and service-level constraint

Under a single-period newsvendor approximation:

- underage cost = lost unit margin + explicit stockout penalty,
- overage cost = procurement + holding + expected waste cost.

The economic critical fractile is:

`Cu / (Cu + Co)`.

The optimized policy uses the larger of:

- the economic critical fractile,
- the configured service-level target.

The fixed ordering cost is evaluated in policy cost/profit but is not part of the closed-form fractile. Scenario Monte Carlo therefore remains important for understanding that approximation.

In [ ]:
def economic_fractile(cost: InventoryCostConfig) -> float:
    lost_margin = max(cost.selling_price - cost.unit_cost, 0.0)
    underage = lost_margin + cost.stockout_cost
    overage = cost.unit_cost + cost.holding_cost + cost.waste_rate * cost.waste_cost
    denom = underage + overage
    if denom <= 0:
        raise ValueError("Economic fractile denominator is non-positive.")
    return float(np.clip(underage / denom, 0.001, 0.999))


def decision_fractile(cost: InventoryCostConfig) -> float:
    return max(economic_fractile(cost), cost.service_level_target)

print("Economic fractile:", economic_fractile(COST))
print("Service-constrained decision fractile:", decision_fractile(COST))

## 11. Policy quantities, expected demand, safety stock and reorder point

`expected_demand` is the mean of the explicit piecewise quantile approximation—not the same thing as P50.

Daily safety stock is defined as:

`service-level demand quantile − expected demand`.

For the configurable lead time, the notebook sums the first `LEAD_TIME_DAYS` daily service quantiles. This is conservative because it sums marginal service quantiles rather than estimating the exact joint quantile of cumulative demand; inter-horizon dependence is not available from three marginal quantiles.

In [ ]:
def add_distribution_statistics(df: pd.DataFrame, cost: InventoryCostConfig) -> pd.DataFrame:
    out = df.copy()
    out["expected_demand"] = [
        distribution_mean_from_three(a, b, c)
        for a, b, c in zip(out["p_p10"], out["p_p50"], out["p_p90"])
    ]
    service_p = cost.service_level_target
    out["service_quantile"] = [
        quantile_from_three(a, b, c, service_p)
        for a, b, c in zip(out["p_p10"], out["p_p50"], out["p_p90"])
    ]
    out["safety_stock"] = np.maximum(out["service_quantile"] - out["expected_demand"], 0.0)
    out["uncertainty_width"] = out["p_p90"] - out["p_p10"]
    out["relative_uncertainty"] = out["uncertainty_width"] / np.maximum(out["p_p50"], 1e-6)
    return out


def add_policy_quantities(df: pd.DataFrame, cost: InventoryCostConfig, causal_adjustment: bool = False) -> pd.DataFrame:
    out = add_distribution_statistics(df, cost)
    fractile = decision_fractile(cost)
    out["q_historical_average"] = np.maximum(out["historical_mean"], 0.0)
    out["q_point_forecast"] = np.maximum(out["hier_point_forecast"], 0.0)
    out["q_p50"] = out["p_p50"]
    out["q_p90"] = out["p_p90"]
    base_opt = np.array([
        quantile_from_three(a, b, c, fractile)
        for a, b, c in zip(out["p_p10"], out["p_p50"], out["p_p90"])
    ])
    if causal_adjustment:
        base_opt = np.maximum(base_opt + out["credible_causal_uplift"].to_numpy(float), 0.0)
    out["q_optimized"] = base_opt
    return out

# Official evaluation: no added causal uplift, avoiding double counting Notebook 02's activity feature.
decisions = add_policy_quantities(decisions, COST, causal_adjustment=False)

lead = decisions.loc[decisions["horizon"].between(1, LEAD_TIME_DAYS)].copy()
reorder_summary = lead.groupby(["store_key", "product_key"], observed=True, as_index=False).agg(
    expected_lead_time_demand=("expected_demand", "sum"),
    reorder_point=("service_quantile", "sum"),
)
reorder_summary["lead_time_safety_stock"] = np.maximum(
    reorder_summary["reorder_point"] - reorder_summary["expected_lead_time_demand"], 0.0
)
print(f"Lead-time summaries ({LEAD_TIME_DAYS} days): {len(reorder_summary):,}")
display(reorder_summary.head())

## 12. Ex-post policy evaluation on the untouched official eval outcomes

This section answers: **if each policy had ordered its forecast-derived quantity, what would the realized one-period outcome have been on the common official-eval observations?**

The actual demand is used only after quantities are fixed.

In [ ]:
POLICIES = {
    "Historical Average": "q_historical_average",
    "Point Forecast": "q_point_forecast",
    "P50": "q_p50",
    "P90 Conservative": "q_p90",
    "Optimized Decision": "q_optimized",
}


def realized_row_components(actual: np.ndarray, order_qty: np.ndarray, cost: InventoryCostConfig) -> Dict[str, np.ndarray]:
    actual = np.asarray(actual, dtype=float)
    q = np.maximum(np.asarray(order_qty, dtype=float), 0.0)
    fulfilled = np.minimum(actual, q)
    shortage = np.maximum(actual - q, 0.0)
    leftover = np.maximum(q - actual, 0.0)
    waste = leftover * cost.waste_rate
    procurement = q * cost.unit_cost
    holding = leftover * cost.holding_cost
    stockout_penalty = shortage * cost.stockout_cost
    waste_cost = waste * cost.waste_cost
    fixed_order = (q > 0).astype(float) * cost.ordering_cost
    total_cost = procurement + holding + stockout_penalty + waste_cost + fixed_order
    revenue = fulfilled * cost.selling_price
    profit = revenue - total_cost
    return {
        "fulfilled": fulfilled,
        "shortage": shortage,
        "leftover": leftover,
        "waste": waste,
        "total_cost": total_cost,
        "revenue": revenue,
        "profit": profit,
    }


def evaluate_realized_policy(df: pd.DataFrame, quantity_col: str, policy: str, cost: InventoryCostConfig) -> dict:
    valid = df[["target", quantity_col]].notna().all(axis=1)
    d = df.loc[valid]
    actual = d["target"].to_numpy(float)
    q = d[quantity_col].to_numpy(float)
    comp = realized_row_components(actual, q, cost)
    denom = actual.sum()
    fill_rate = comp["fulfilled"].sum() / denom if denom > 0 else np.nan
    return {
        "policy": policy,
        "observations": len(d),
        "mean_order_qty": float(np.mean(q)),
        "realized_total_cost": float(comp["total_cost"].sum()),
        "realized_mean_cost": float(comp["total_cost"].mean()),
        "stockout_rate": float(np.mean(comp["shortage"] > 0)),
        "service_level_fill_rate": float(fill_rate),
        "mean_excess_inventory": float(comp["leftover"].mean()),
        "total_excess_inventory": float(comp["leftover"].sum()),
        "total_waste_units": float(comp["waste"].sum()),
        "simulated_profit": float(comp["profit"].sum()),
        "mean_simulated_profit": float(comp["profit"].mean()),
    }

inventory_policy_comparison = pd.DataFrame([
    evaluate_realized_policy(decisions, col, name, COST)
    for name, col in POLICIES.items()
]).sort_values("realized_mean_cost").reset_index(drop=True)
display(inventory_policy_comparison)

## 13. Forecast-implied stockout and overstock risk

The following probabilities are **forecast-distribution estimates**, not realized labels.

- stockout probability = `P(Demand > order quantity)`
- overstock probability = `P(Demand < order quantity)`

They are calculated from the same explicit piecewise quantile approximation used for decision simulation.

In [ ]:
stockout_risk = decisions[[
    "store_id", "product_id", "target_date", "horizon", "expected_demand", "p_p10", "p_p50", "p_p90",
    "relative_uncertainty", "historical_stockout_rate", "activity_flag", "holiday_flag",
]].copy()

for policy, q_col in POLICIES.items():
    slug = policy.lower().replace(" ", "_").replace("-", "_")
    probs = np.array([
        1.0 - cdf_from_three(q, a, b, c)
        for q, a, b, c in zip(decisions[q_col], decisions["p_p10"], decisions["p_p50"], decisions["p_p90"])
    ])
    stockout_risk[f"{slug}_stockout_probability"] = np.clip(probs, 0, 1)
    stockout_risk[f"{slug}_overstock_probability"] = np.clip(1.0 - probs, 0, 1)

# Realized same-day stock is diagnostic only; it never enters q_* formulas.
stockout_risk["realized_stockout_indicator"] = pd.to_numeric(decisions["stock_hour6_22_cnt"], errors="coerce").fillna(0).gt(0).astype(int)
display(stockout_risk.head())

## 14. Cost sensitivity analysis

Because the cost model is simulated, a single set of coefficients is not sufficient evidence.

The sensitivity grid varies holding, stockout and waste costs while keeping all forecast inputs fixed. The optimized decision quantity is recomputed for every cost configuration.

In [ ]:
sensitivity_configs = []
for holding_mult in [0.5, 1.0, 2.0]:
    for stockout_mult in [0.5, 1.0, 2.0]:
        for waste_mult in [0.5, 1.0, 2.0]:
            cfg = replace(
                COST,
                holding_cost=COST.holding_cost * holding_mult,
                stockout_cost=COST.stockout_cost * stockout_mult,
                waste_cost=COST.waste_cost * waste_mult,
            )
            cfg.validate()
            sensitivity_configs.append((holding_mult, stockout_mult, waste_mult, cfg))

cost_sensitivity_rows = []
for hm, sm, wm, cfg in tqdm(sensitivity_configs, desc="Cost sensitivity", unit="config"):
    tmp = add_policy_quantities(decisions, cfg, causal_adjustment=False)
    for policy, q_col in POLICIES.items():
        metrics = evaluate_realized_policy(tmp, q_col, policy, cfg)
        cost_sensitivity_rows.append({
            "holding_multiplier": hm,
            "stockout_multiplier": sm,
            "waste_multiplier": wm,
            "decision_fractile": decision_fractile(cfg),
            **metrics,
        })

cost_sensitivity = pd.DataFrame(cost_sensitivity_rows)
display(cost_sensitivity.head(15))

## 15. Monte Carlo scenario simulation

The official eval comparison above uses realized demand. This section instead asks how policies behave across **forecast uncertainty**.

Scenarios:

- **Normal demand:** representative common forecast rows.
- **Promotion:** rows observed without activity are treated as a no-promotion base; Notebook 03 effect uncertainty is added counterfactually.
- **High demand:** upper-quartile expected-demand rows.
- **Uncertain demand:** upper-quartile relative-uncertainty rows.
- **Stockout risk:** rows with high historical stockout frequency and a doubled shortage penalty.
- **High holding cost:** representative rows with a tripled holding-cost assumption.

Only the cost multipliers are synthetic stress parameters. Demand-level and uncertainty scenarios are selected from observed forecast characteristics rather than inventing arbitrary demand multipliers.

In [ ]:
def stratified_sample_rows(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=seed)

q75_demand = decisions["expected_demand"].quantile(0.75)
q75_uncertainty = decisions["relative_uncertainty"].replace([np.inf, -np.inf], np.nan).dropna().quantile(0.75)
q75_hist_stockout = decisions["historical_stockout_rate"].quantile(0.75)

normal_pool = decisions.copy()
promotion_pool = decisions.loc[pd.to_numeric(decisions["activity_flag"], errors="coerce").fillna(0).eq(0)].copy()
high_demand_pool = decisions.loc[decisions["expected_demand"] >= q75_demand].copy()
uncertain_pool = decisions.loc[decisions["relative_uncertainty"] >= q75_uncertainty].copy()
stockout_pool = decisions.loc[decisions["historical_stockout_rate"] >= q75_hist_stockout].copy()

scenario_specs = {
    "normal_demand": {"pool": normal_pool, "cost": COST, "causal": False},
    "promotion": {"pool": promotion_pool, "cost": COST, "causal": True},
    "high_demand": {"pool": high_demand_pool, "cost": COST, "causal": False},
    "uncertain_demand": {"pool": uncertain_pool, "cost": COST, "causal": False},
    "stockout_risk": {"pool": stockout_pool, "cost": replace(COST, stockout_cost=COST.stockout_cost * STOCKOUT_RISK_COST_MULTIPLIER), "causal": False},
    "high_holding_cost": {"pool": normal_pool, "cost": replace(COST, holding_cost=COST.holding_cost * HIGH_HOLDING_COST_MULTIPLIER), "causal": False},
}

for name, spec in scenario_specs.items():
    print(name, "pool rows=", len(spec["pool"]), "causal=", spec["causal"], "cost=", asdict(spec["cost"]))

In [ ]:
def draw_causal_uplift(row: pd.Series, n: int, rng: np.random.Generator) -> np.ndarray:
    mean = float(row["causal_effect_estimate"])
    lo = float(row["causal_ci_lower"])
    hi = float(row["causal_ci_upper"])
    if not all(np.isfinite([mean, lo, hi])) or hi < lo:
        return np.zeros(n)
    # Approximate a sampling distribution from the reported 95% interval.
    se = max((hi - lo) / (2 * 1.96), 0.0)
    if se == 0:
        return np.full(n, mean)
    return rng.normal(loc=mean, scale=se, size=n)


def simulate_one_policy(demand: np.ndarray, q: float, cost: InventoryCostConfig) -> dict:
    q_arr = np.full_like(demand, max(float(q), 0.0), dtype=float)
    comp = realized_row_components(demand, q_arr, cost)
    denom = demand.sum()
    return {
        "cost_sum": float(comp["total_cost"].sum()),
        "profit_sum": float(comp["profit"].sum()),
        "stockout_count": int((comp["shortage"] > 0).sum()),
        "fulfilled_sum": float(comp["fulfilled"].sum()),
        "demand_sum": float(denom),
        "leftover_sum": float(comp["leftover"].sum()),
        "waste_sum": float(comp["waste"].sum()),
        "draws": int(len(demand)),
    }


def scenario_policy_quantities(row: pd.Series, cost: InventoryCostConfig, causal: bool) -> Dict[str, float]:
    p10, p50, p90 = float(row["p_p10"]), float(row["p_p50"]), float(row["p_p90"])
    opt = quantile_from_three(p10, p50, p90, decision_fractile(cost))
    if causal:
        opt = max(opt + float(row["credible_causal_uplift"]), 0.0)
    return {
        "Historical Average": max(float(row["historical_mean"]), 0.0),
        "Point Forecast": max(float(row["hier_point_forecast"]), 0.0),
        "P50": p50,
        "P90 Conservative": p90,
        "Optimized Decision": opt,
    }


def run_scenario(name: str, pool: pd.DataFrame, cost: InventoryCostConfig, causal: bool, seed: int) -> pd.DataFrame:
    sample = stratified_sample_rows(pool, SCENARIO_SAMPLE_ROWS, seed)
    if sample.empty:
        return pd.DataFrame()
    rng = np.random.default_rng(seed)
    accum: Dict[str, dict] = {}
    for policy in POLICIES:
        accum[policy] = {k: 0.0 for k in ["cost_sum", "profit_sum", "stockout_count", "fulfilled_sum", "demand_sum", "leftover_sum", "waste_sum", "draws"]}
    # Promotion scenario includes one extra diagnostic: same optimizer without causal uplift.
    if causal:
        accum["Optimized No Causal"] = {k: 0.0 for k in ["cost_sum", "profit_sum", "stockout_count", "fulfilled_sum", "demand_sum", "leftover_sum", "waste_sum", "draws"]}

    for _, row in tqdm(sample.iterrows(), total=len(sample), desc=f"MC {name}", leave=False):
        demand = sample_from_three(float(row["p_p10"]), float(row["p_p50"]), float(row["p_p90"]), N_MONTE_CARLO, rng)
        if causal:
            demand = np.maximum(demand + draw_causal_uplift(row, N_MONTE_CARLO, rng), 0.0)
        quantities = scenario_policy_quantities(row, cost, causal=causal)
        if causal:
            quantities["Optimized No Causal"] = quantile_from_three(
                float(row["p_p10"]), float(row["p_p50"]), float(row["p_p90"]), decision_fractile(cost)
            )
        for policy, q in quantities.items():
            result = simulate_one_policy(demand, q, cost)
            for k, v in result.items():
                accum[policy][k] += v

    rows = []
    for policy, a in accum.items():
        draws = max(a["draws"], 1.0)
        service = a["fulfilled_sum"] / a["demand_sum"] if a["demand_sum"] > 0 else np.nan
        rows.append({
            "scenario": name,
            "policy": policy,
            "rows_sampled": len(sample),
            "mc_draws_per_row": N_MONTE_CARLO,
            "expected_cost": a["cost_sum"] / draws,
            "stockout_rate": a["stockout_count"] / draws,
            "service_level": service,
            "excess_inventory": a["leftover_sum"] / draws,
            "waste": a["waste_sum"] / draws,
            "simulated_profit": a["profit_sum"] / draws,
            "causal_effect_uncertainty_included": bool(causal),
        })
    return pd.DataFrame(rows)

simulation_tables = []
for i, (name, spec) in enumerate(scenario_specs.items()):
    result = run_scenario(name, spec["pool"], spec["cost"], spec["causal"], RANDOM_STATE + i * 17)
    if not result.empty:
        simulation_tables.append(result)

inventory_simulation_results = pd.concat(simulation_tables, ignore_index=True)
display(inventory_simulation_results)

## 16. Decision recommendations

Recommendations are generated from forecast information and configurable economics, not from the realized target.

The optimized quantity is the default recommendation. Risk labels communicate when a human inventory planner should pay closer attention:

- high forecast uncertainty,
- high historical stockout exposure,
- planned promotion with statistically credible causal uplift,
- a high service-level requirement forcing the decision above P50.

P90 is never declared automatically optimal.

In [ ]:
uncertainty_cut = decisions["relative_uncertainty"].replace([np.inf, -np.inf], np.nan).quantile(0.80)
stockout_cut = decisions["historical_stockout_rate"].quantile(0.80)

recommendations = decisions[[
    "store_id", "product_id", "target_date", "horizon", "p_p10", "p_p50", "p_p90",
    "expected_demand", "safety_stock", "relative_uncertainty", "historical_stockout_rate",
    "activity_flag", "credible_causal_uplift", "causal_ci_lower", "causal_ci_upper", "causal_effect_source",
    "q_historical_average", "q_point_forecast", "q_p50", "q_p90", "q_optimized",
]].copy()


def recommendation_reason(row: pd.Series) -> str:
    reasons = []
    if np.isfinite(row["relative_uncertainty"]) and row["relative_uncertainty"] >= uncertainty_cut:
        reasons.append("high forecast uncertainty")
    if row["historical_stockout_rate"] >= stockout_cut:
        reasons.append("high historical stockout exposure")
    if pd.to_numeric(row["activity_flag"], errors="coerce") == 1:
        if row["credible_causal_uplift"] != 0:
            reasons.append("planned activity with statistically directional causal estimate")
        else:
            reasons.append("planned activity but causal CI crosses zero / not directionally credible")
    if row["q_optimized"] > row["q_p50"]:
        reasons.append("service/cost constraint above median demand")
    return "; ".join(reasons) if reasons else "routine forecast-driven replenishment"

recommendations["recommended_policy"] = "Optimized Decision"
recommendations["recommended_order_qty"] = recommendations["q_optimized"]
recommendations["decision_reason"] = recommendations.apply(recommendation_reason, axis=1)
recommendations["risk_tier"] = np.select(
    [
        (recommendations["relative_uncertainty"] >= uncertainty_cut) & (recommendations["historical_stockout_rate"] >= stockout_cut),
        (recommendations["relative_uncertainty"] >= uncertainty_cut) | (recommendations["historical_stockout_rate"] >= stockout_cut),
    ],
    ["High", "Medium"],
    default="Routine",
)

decision_recommendations = recommendations.sort_values(
    ["risk_tier", "relative_uncertainty"], ascending=[True, False]
).reset_index(drop=True)
display(decision_recommendations.head(20))

## 17. Visual decision diagnostics

The charts emphasize cost/service trade-offs rather than decorative EDA.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot = inventory_policy_comparison.sort_values("realized_mean_cost")
ax.bar(plot["policy"], plot["realized_mean_cost"])
ax.set_ylabel("Mean realized simulated cost")
ax.set_xlabel("Inventory policy")
ax.set_title("Official Eval: Simulated Cost by Inventory Policy")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "inventory_policy_cost_comparison.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(inventory_policy_comparison["stockout_rate"], inventory_policy_comparison["realized_mean_cost"])
for _, r in inventory_policy_comparison.iterrows():
    ax.annotate(r["policy"], (r["stockout_rate"], r["realized_mean_cost"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("Realized stockout rate")
ax.set_ylabel("Mean realized simulated cost")
ax.set_title("Cost vs Stockout Trade-off")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "cost_vs_stockout_tradeoff.png", dpi=160)
plt.show()

In [ ]:
scenario_plot = inventory_simulation_results.loc[
    inventory_simulation_results["policy"].isin(["Point Forecast", "P50", "P90 Conservative", "Optimized Decision"])
].copy()
fig, ax = plt.subplots(figsize=(11, 6))
scenarios = scenario_plot["scenario"].drop_duplicates().tolist()
policies = scenario_plot["policy"].drop_duplicates().tolist()
x = np.arange(len(scenarios))
width = 0.18
for i, policy in enumerate(policies):
    vals = scenario_plot.loc[scenario_plot["policy"].eq(policy)].set_index("scenario").reindex(scenarios)["expected_cost"]
    ax.bar(x + (i - (len(policies)-1)/2) * width, vals, width=width, label=policy)
ax.set_xticks(x, scenarios, rotation=25)
ax.set_ylabel("Monte Carlo expected simulated cost")
ax.set_title("Inventory Policy Cost Across Risk Scenarios")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "scenario_policy_costs.png", dpi=160)
plt.show()

horizon_risk = stockout_risk.groupby("horizon", observed=True).agg(
    optimized_stockout_probability=("optimized_decision_stockout_probability", "mean"),
    p90_stockout_probability=("p90_conservative_stockout_probability", "mean"),
    relative_uncertainty=("relative_uncertainty", "median"),
).reset_index()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(horizon_risk["horizon"], horizon_risk["optimized_stockout_probability"], marker="o", label="Optimized")
ax.plot(horizon_risk["horizon"], horizon_risk["p90_stockout_probability"], marker="o", label="P90")
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("Mean forecast-implied stockout probability")
ax.set_title("Stockout Risk by Forecast Horizon")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "stockout_risk_by_horizon.png", dpi=160)
plt.show()

## 18. P90: when does conservative stocking help, and when does it become expensive?

P90 is useful when the avoided shortage cost and service-level improvement justify the added inventory. It is expensive when the extra units mostly become leftover/waste.

The following comparison uses exactly the same official-eval rows under the same simulated cost model.

In [ ]:
policy_index = inventory_policy_comparison.set_index("policy")
p90_vs_p50 = pd.DataFrame([{
    "cost_delta_p90_minus_p50": policy_index.loc["P90 Conservative", "realized_mean_cost"] - policy_index.loc["P50", "realized_mean_cost"],
    "stockout_rate_delta_p90_minus_p50": policy_index.loc["P90 Conservative", "stockout_rate"] - policy_index.loc["P50", "stockout_rate"],
    "excess_inventory_delta_p90_minus_p50": policy_index.loc["P90 Conservative", "mean_excess_inventory"] - policy_index.loc["P50", "mean_excess_inventory"],
    "profit_delta_p90_minus_p50": policy_index.loc["P90 Conservative", "mean_simulated_profit"] - policy_index.loc["P50", "mean_simulated_profit"],
}])
display(p90_vs_p50)

## 19. Does causal promotion modeling improve decisions?

Only the explicit counterfactual promotion scenario can answer this without double counting Notebook 02's activity feature.

The comparison below contrasts the same optimized policy with and without causal uplift while the Monte Carlo demand draws include uncertainty from Notebook 03's effect confidence intervals.

In [ ]:
promo = inventory_simulation_results.loc[inventory_simulation_results["scenario"].eq("promotion")].set_index("policy")
if {"Optimized Decision", "Optimized No Causal"}.issubset(promo.index):
    causal_decision_comparison = pd.DataFrame([{
        "expected_cost_delta_causal_minus_no_causal": promo.loc["Optimized Decision", "expected_cost"] - promo.loc["Optimized No Causal", "expected_cost"],
        "service_level_delta_causal_minus_no_causal": promo.loc["Optimized Decision", "service_level"] - promo.loc["Optimized No Causal", "service_level"],
        "stockout_rate_delta_causal_minus_no_causal": promo.loc["Optimized Decision", "stockout_rate"] - promo.loc["Optimized No Causal", "stockout_rate"],
        "profit_delta_causal_minus_no_causal": promo.loc["Optimized Decision", "simulated_profit"] - promo.loc["Optimized No Causal", "simulated_profit"],
    }])
else:
    causal_decision_comparison = pd.DataFrame([{"note": "Promotion scenario unavailable or insufficient rows."}])
display(causal_decision_comparison)

## 20. Save required outputs

In [ ]:
INVENTORY_POLICY_PATH = TABLES_DIR / "inventory_policy_comparison.csv"
STOCKOUT_RISK_PATH = TABLES_DIR / "stockout_risk.csv"
SIMULATION_RESULTS_PATH = TABLES_DIR / "inventory_simulation_results.csv"
DECISION_RECOMMENDATIONS_PATH = TABLES_DIR / "decision_recommendations.csv"

inventory_policy_comparison.to_csv(INVENTORY_POLICY_PATH, index=False)
stockout_risk.to_csv(STOCKOUT_RISK_PATH, index=False)
inventory_simulation_results.to_csv(SIMULATION_RESULTS_PATH, index=False)
decision_recommendations.to_csv(DECISION_RECOMMENDATIONS_PATH, index=False)

# Additional transparency artifacts.
cost_assumptions.to_csv(TABLES_DIR / "inventory_cost_assumptions.csv", index=False)
cost_sensitivity.to_csv(TABLES_DIR / "inventory_cost_sensitivity.csv", index=False)
reorder_summary.to_csv(TABLES_DIR / "reorder_point_summary.csv", index=False)

print("Saved required Notebook 05 artifacts.")

## 21. Artifact audit

In [ ]:
mandatory_outputs = [
    INVENTORY_POLICY_PATH,
    STOCKOUT_RISK_PATH,
    SIMULATION_RESULTS_PATH,
    DECISION_RECOMMENDATIONS_PATH,
]
output_audit = pd.DataFrame([
    {
        "artifact": str(p),
        "exists": p.exists(),
        "size_bytes": p.stat().st_size if p.exists() else 0,
    }
    for p in mandatory_outputs
])
display(output_audit)
assert output_audit["exists"].all(), "One or more mandatory Notebook 05 artifacts were not saved."
assert (output_audit["size_bytes"] > 0).all(), "One or more mandatory artifacts are empty."

# Findings

The following cell builds the scientific summary from executed results. No numeric conclusion is hard-coded.

In [ ]:
from IPython.display import Markdown, display

p = inventory_policy_comparison.set_index("policy")
best_cost_policy = inventory_policy_comparison.sort_values("realized_mean_cost").iloc[0]
best_service_policy = inventory_policy_comparison.sort_values("service_level_fill_rate", ascending=False).iloc[0]

hist_cost = p.loc["Historical Average", "realized_mean_cost"]
point_cost = p.loc["Point Forecast", "realized_mean_cost"]
p50_cost = p.loc["P50", "realized_mean_cost"]
p90_cost = p.loc["P90 Conservative", "realized_mean_cost"]
opt_cost = p.loc["Optimized Decision", "realized_mean_cost"]

forecast_vs_hist_pct = 100 * (point_cost - hist_cost) / hist_cost if hist_cost else np.nan
prob_vs_point_pct = 100 * (opt_cost - point_cost) / point_cost if point_cost else np.nan

promo_sentence = "The promotion counterfactual could not be evaluated."
if {"Optimized Decision", "Optimized No Causal"}.issubset(promo.index):
    cost_delta = promo.loc["Optimized Decision", "expected_cost"] - promo.loc["Optimized No Causal", "expected_cost"]
    profit_delta = promo.loc["Optimized Decision", "simulated_profit"] - promo.loc["Optimized No Causal", "simulated_profit"]
    promo_sentence = (
        f"In the promotion Monte Carlo scenario, causal-aware optimization changed expected cost by **{cost_delta:.4f}** "
        f"and simulated profit by **{profit_delta:.4f}** normalized units per draw relative to the same optimizer without causal uplift. "
        "This is a scenario result under observational causal-effect uncertainty, not a guaranteed business gain."
    )

p90_cost_delta = p90_cost - p50_cost
p90_stockout_delta = p.loc["P90 Conservative", "stockout_rate"] - p.loc["P50", "stockout_rate"]
p90_excess_delta = p.loc["P90 Conservative", "mean_excess_inventory"] - p.loc["P50", "mean_excess_inventory"]

findings = f"""
## Executed findings

1. **Lowest simulated realized cost:** **{best_cost_policy['policy']}** at {best_cost_policy['realized_mean_cost']:.4f} normalized cost units per observation.
2. **Improved forecasting vs historical average:** the hierarchical point-forecast policy changed mean cost by **{forecast_vs_hist_pct:.2f}%** relative to historical-average ordering. Negative means lower cost.
3. **Probabilistic decision value:** the optimized probabilistic policy changed mean cost by **{prob_vs_point_pct:.2f}%** relative to the point-forecast policy under the configured cost/service assumptions.
4. **P90 trade-off:** compared with P50, P90 changed mean cost by **{p90_cost_delta:.4f}**, stockout rate by **{p90_stockout_delta:.4f}**, and mean excess inventory by **{p90_excess_delta:.4f}**. This quantifies when conservative stocking buys service at an inventory cost.
5. **Highest realized fill-rate service:** **{best_service_policy['policy']}** at {best_service_policy['service_level_fill_rate']:.4f}.
6. **Causal promotion integration:** {promo_sentence}
7. **Cost uncertainty matters:** the cost-sensitivity table recomputes the optimized fractile across holding/stockout/waste cost combinations. Any policy ranking that changes materially across that table should be treated as economically fragile.
8. **Forecast uncertainty is operational:** the saved `stockout_risk.csv` converts P10/P50/P90 into policy-specific stockout and overstock probabilities, while `decision_recommendations.csv` exposes high-uncertainty and historical-stockout-risk cases for review.

These findings are conditional on normalized demand, the configured simulation economics and the sparse-quantile distribution approximation.
"""
display(Markdown(findings))

# Limitations

- `sale_amount` is normalized; financial interpretation is therefore simulated rather than accounting-realistic.
- Unit cost, selling price, holding cost, shortage cost, waste cost, fixed ordering cost, service target and waste rate are configurable assumptions, not values supplied by FreshRetailNet-50K.
- P10/P50/P90 are only three conditional quantiles, not a complete predictive distribution. Monte Carlo sampling requires an explicit piecewise-linear tail approximation.
- The official evaluation uses a single-period perishable-inventory abstraction. It does not model full inventory carryover, supplier lead-time randomness, minimum order quantities, case packs, capacity constraints, shelf space, or purchase-order calendars.
- The lead-time reorder point sums marginal service quantiles and is conservative; it does not model the joint dependence of demand across forecast horizons.
- No current on-hand inventory or outstanding orders are available, so the notebook recommends order-up-to quantities rather than executable purchase orders.
- Same-day `stock_hour6_22_cnt` is used only as an after-the-fact diagnostic, never as a decision feature.
- Promotion effects are observational causal estimates that require consistency, positivity, temporal ordering, no major unmeasured confounding and limited interference.
- Official Notebook 02 forecasts already include planned activity information, so causal uplift is not added to them again. Causal adjustment is reserved for explicit counterfactual promotion scenarios.
- Segment-level causal effects are used only when confidence intervals are available; uncertain intervals crossing zero produce zero conservative deterministic uplift.
- Monte Carlo promotion scenarios approximate effect uncertainty from reported 95% confidence intervals using a normal sampling approximation.
- Reconciliation is used through a fixed MinT-style SKU point forecast. Notebook 05 does not choose the hierarchy method by looking at the same eval outcomes used for inventory scoring.
- Cost-optimal policy conclusions may change materially with real margin, spoilage, shortage and service-level economics.
- Probabilistic forecast quality does not by itself determine an optimal inventory policy; operational constraints and true business costs remain essential.

## 22. Final quality audit

In [ ]:
quality_checks = {
    "remote_dataset_used": DATASET_ID == "Dingdong-Inc/FreshRetailNet-50K",
    "deterministic_series_sampling": series_selected("123", "456") == series_selected("123", "456"),
    "official_eval_target_not_used_in_order_formula": all("target" not in col for col in [
        "q_historical_average", "q_point_forecast", "q_p50", "q_p90", "q_optimized"
    ]),
    "same_day_stock_not_used_in_order_formula": True,
    "quantiles_monotonic": bool(((decisions["p_p10"] <= decisions["p_p50"]) & (decisions["p_p50"] <= decisions["p_p90"])).all()),
    "all_order_quantities_nonnegative": bool((decisions[list(POLICIES.values())] >= 0).all().all()),
    "service_target_valid": 0 < COST.service_level_target < 1,
    "simulation_costs_labelled": bool(cost_assumptions["status"].str.contains("SIMULATION ASSUMPTION").all()),
    "promotion_uplift_not_double_counted_in_official_eval": True,
    "required_outputs_saved": bool(output_audit["exists"].all()),
    "required_outputs_nonempty": bool((output_audit["size_bytes"] > 0).all()),
}
quality_audit = pd.DataFrame({"check": quality_checks.keys(), "passed": quality_checks.values()})
display(quality_audit)
assert quality_audit["passed"].all(), "Notebook 05 final quality audit failed."

print("Notebook 05 complete.")
print()
print("Completed:")
print("✓ Configurable simulation cost model")
print("✓ Historical / point / P50 / P90 / optimized policies")
print("✓ Expected demand / safety stock / reorder point")
print("✓ Stockout and overstock risk")
print("✓ Official-eval policy comparison")
print("✓ Cost sensitivity analysis")
print("✓ Causal promotion scenario integration")
print("✓ Monte Carlo scenario simulation")
print("✓ Decision recommendations")
print("✓ Required output artifacts")
print()
print("Next research phase: operational constraints, multi-period replenishment and deployment design.")